<a href="https://colab.research.google.com/github/varunsmn87/SolarcycleDTW/blob/main/DTW_upload_ver3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# §1 SETUP

!pip -q install fastdtw scikit-learn scipy matplotlib pandas numpy

import os, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.cm as cm

from fastdtw import fastdtw
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Global constants
# ------------------------------------------------------------------
CUT_OFF    = pd.Timestamp("2024-12-01")   # data freeze date
RNG_SEED   = 42
PREFIX_LEN = 60    # observed months used for real-time mode
BINS       = 12    # phase bins for residual bootstrap
N_BOOT     = 5000  # bootstrap draws per forecast month
NU_PUB     = 0.35  # published variance floor

np.random.seed(RNG_SEED)

OUT_FIG = "results/figures"
OUT_TAB = "results/tables"
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TAB, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "font.size": 10,   "axes.titlesize": 11,
    "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9,  "ytick.labelsize": 9,
    "axes.grid": True,  "grid.alpha": 0.25,
})

STAMP = f"Data ≤ {CUT_OFF.date()} | RNG={RNG_SEED}"

def true_rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))

def safe_save(fig, stem):
    """Save figure as PDF and EPS for journal submission."""
    fig.tight_layout()
    for ext in ("pdf", "eps"):
        fig.savefig(os.path.join(OUT_FIG, f"{stem}.{ext}"), bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {stem}.pdf / .eps")

print(f"Setup complete. STAMP = '{STAMP}'")
print(f"Output dirs: {OUT_FIG}/, {OUT_TAB}/")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Setup complete. STAMP = 'Data ≤ 2024-12-01 | RNG=42'
Output dirs: results/figures/, results/tables/


In [ ]:

# §2  DATA LOADING AND CYCLE SEGMENTATION
#
# Source: SILSO v2.0 monthly smoothed sunspot number
#         Royal Observatory of Belgium
#         https://www.sidc.be/SILSO/DATA/SN_ms_tot_V2.0.csv


FROZEN_DIR = "data_frozen"
os.makedirs(FROZEN_DIR, exist_ok=True)
FROZEN_CSV = os.path.join(
    FROZEN_DIR,
    f"SILSO_SN_ms_tot_V2_until_{CUT_OFF.date()}.csv"
)
SILSO_URL = "https://www.sidc.be/SILSO/DATA/SN_ms_tot_V2.0.csv"

if os.path.exists(FROZEN_CSV):
    df = pd.read_csv(FROZEN_CSV, parse_dates=["Date"]).set_index("Date")
    print(f"Loaded frozen SILSO snapshot: {len(df)} rows  "
          f"({df.index[0].date()} → {df.index[-1].date()})")
else:
    print("First-run: downloading SILSO v2.0 and freezing at "
          f"{CUT_OFF.date()} …")
    raw = pd.read_csv(SILSO_URL, header=None, delimiter=";")
    raw.columns = ["Year", "Month", "DateFraction", "SmoothedSSN",
                   "Definitive", "Extra1", "Extra2"]
    raw["Date"] = pd.to_datetime(
        raw["Year"].astype(str) + "-" + raw["Month"].astype(str),
        errors="coerce")
    df = raw.dropna(subset=["Date"]).set_index("Date")
    df = df[df["SmoothedSSN"] >= 0].loc[:CUT_OFF].copy()
    df.reset_index().to_csv(FROZEN_CSV, index=False)
    print(f"Snapshot frozen → {FROZEN_CSV}")
    print(f"  {len(df)} rows  "
          f"({df.index[0].date()} → {df.index[-1].date()})")
    print("  All subsequent runs will use this local file.")

assert df.index[-1] <= CUT_OFF, (
    f"Data extends past freeze date {CUT_OFF.date()} — "
    "delete the frozen CSV and rerun to regenerate it."
)

# ------------------------------------------------------------------
# Cycle boundary dates from published SILSO minimum catalogue
# (Clette & Lefevre 2015; Hathaway 2015)
# ------------------------------------------------------------------
cycle_boundaries = {
    13: ("1889-03-01", "1901-02-01"),
    14: ("1901-03-01", "1913-07-01"),
    15: ("1913-08-01", "1923-07-01"),
    16: ("1923-08-01", "1933-02-01"),
    17: ("1933-03-01", "1944-01-01"),
    18: ("1944-02-01", "1954-03-01"),
    19: ("1954-04-01", "1964-09-01"),
    20: ("1964-10-01", "1976-02-01"),
    21: ("1976-03-01", "1986-08-01"),
    22: ("1986-09-01", "1996-07-01"),
    23: ("1996-08-01", "2008-12-01"),
    24: ("2009-01-01", "2019-12-01"),
}

cycle_fits = {}
print("\nCycle library (SC13–SC24):")
for sc, (st, en) in cycle_boundaries.items():
    seg = df.loc[pd.to_datetime(st):pd.to_datetime(en),
                 "SmoothedSSN"].values.astype(float)
    cycle_fits[sc] = (np.arange(len(seg)), seg)
    print(f"  SC{sc:2d}: {len(seg):3d} months  "
          f"peak={seg.max():6.1f}  min={seg.min():5.1f}")

# Parity-restricted pools
odd_pool  = [c for c in cycle_fits if c % 2 == 1]   # SC13,15,17,19,21,23
even_pool = [c for c in cycle_fits if c % 2 == 0]   # SC14,16,18,20,22,24

# SC25 observed prefix
sc25_start = pd.to_datetime(cycle_boundaries[24][1]) + pd.offsets.MonthBegin(1)
y_obs_25   = df.loc[sc25_start:CUT_OFF, "SmoothedSSN"].values.astype(float)
t_len      = len(y_obs_25)

print(f"\nOdd  pool : {odd_pool}")
print(f"Even pool : {even_pool}")
print(f"SC25 start: {sc25_start.date()}  |  observed months: {t_len}")
print(f"SC25 last observed: {df.loc[sc25_start:CUT_OFF].index[-1].date()}")

First-run: downloading SILSO v2.0 and freezing at 2024-12-01 …
Snapshot frozen → data_frozen/SILSO_SN_ms_tot_V2_until_2024-12-01.csv
  3306 rows  (1749-07-01 → 2024-12-01)
  All subsequent runs will use this local file.

Cycle library (SC13–SC24):
  SC13: 144 months  peak= 146.5  min=  7.3
  SC14: 149 months  peak= 107.1  min=  2.5
  SC15: 120 months  peak= 175.7  min=  2.5
  SC16: 115 months  peak= 130.2  min=  9.4
  SC17: 131 months  peak= 198.6  min=  5.8
  SC18: 122 months  peak= 218.7  min=  6.3
  SC19: 126 months  peak= 285.0  min=  5.1
  SC20: 137 months  peak= 156.6  min= 14.3
  SC21: 126 months  peak= 232.9  min= 14.3
  SC22: 119 months  peak= 212.5  min= 11.2
  SC23: 149 months  peak= 180.3  min=  2.2
  SC24: 132 months  peak= 116.4  min=  1.8

Odd  pool : [13, 15, 17, 19, 21, 23]
Even pool : [14, 16, 18, 20, 22, 24]
SC25 start: 2020-01-01  |  observed months: 60
SC25 last observed: 2024-12-01


In [ ]:
# ============================================================
# §3  PREPROCESSING HELPERS

# ============================================================

def _norm01(y):
    """Min-max normalisation to [0,1]."""
    y = np.asarray(y, float)
    lo, hi = float(np.min(y)), float(np.max(y))
    if hi - lo < 1e-12:
        return np.zeros_like(y)
    return (y - lo) / (hi - lo)

def baseline_reference_curve(n):
    """Half-sinusoidal reference curve for baseline mode"""
    return np.sin(np.pi * np.arange(n) / max(1, n - 1))

In [ ]:
# ============================================================
# §4  CORE DTW ENGINE
#
# DTW distance —
# Implemented via fastdtw (Sakoe & Chiba 1978; Salvador & Chan 2007).
# Both sequences are amplitude-normalised before DTW so that the
# distance reflects morphological shape only, not magnitude.
# Inverse-distance weights
# Normalised template —
# ============================================================

def dtw_select(target_curve, pool_cycles, k):
    """
    Rank pool_cycles by DTW distance to target_curve.
    Both sequences are amplitude-normalised before comparison.
    Each candidate is linearly interpolated to the same length as
    the target so they share a consistent discrete time grid.

    Returns
    -------
    sel       : list of k selected cycle numbers
    w         : normalised IDW weights
    dist_list : full ranked list of (cycle, distance) pairs
    """
    t = _norm01(target_curve)
    dist_list = []
    for c in pool_cycles:
        yc     = _norm01(cycle_fits[c][1])
        yc_adj = np.interp(
            np.linspace(0, 1, len(t)),
            np.linspace(0, 1, len(yc)), yc)
        d, _ = fastdtw(t, yc_adj)
        dist_list.append((c, float(d)))
    dist_list.sort(key=lambda z: z[1])
    chosen = dist_list[:min(k, len(dist_list))]
    sel    = [c for c, _ in chosen]
    eps    = 1e-9   # numerical stability (manuscript Section 3.3)
    w      = np.array([1.0 / (d + eps) for _, d in chosen], float)
    w     /= w.sum()   # satisfy Eq. (5): weights sum to 1
    return sel, w, dist_list


def blend_shape_norm(selected_cycles, w, target_len):
    """
    IDW-weighted average of amplitude-normalised analog profiles.
    Produces the normalised template M(t)
    """
    M = []
    for c in selected_cycles:
        yc = _norm01(cycle_fits[c][1])
        M.append(np.interp(
            np.linspace(0, 1, target_len),
            np.linspace(0, 1, len(yc)), yc))
    return w @ np.vstack(M)   # M(t) in [0,1]


def apply_idw_scaling(shape01, selected_cycles, w):
    """
    Convert normalised template to physical SSN units using
    IDW-weighted analog amplitudes and baselines
    Used in real-time mode.
    """
    amps  = np.array([float(cycle_fits[c][1].max() -
                            cycle_fits[c][1].min())
                      for c in selected_cycles])
    bases = np.array([float(cycle_fits[c][1].min())
                      for c in selected_cycles])
    return np.clip(shape01, 0, 1) * np.sum(w * amps) + np.sum(w * bases)


def apply_climatology_scaling(shape01, pool_cycles):
    """
    Convert normalised template using median peak and minimum
    of the parity-restricted pool.  Used in baseline mode.
    """
    peaks = [float(cycle_fits[c][1].max()) for c in pool_cycles]
    bases = [float(cycle_fits[c][1].min()) for c in pool_cycles]
    return float(np.median(bases)) + np.clip(shape01, 0, 1) * (
           float(np.median(peaks)) - float(np.median(bases)))


print("DTW engine defined.")

DTW engine defined.


In [ ]:
# ============================================================
# §5  FORECAST BUILDERS
#
# Both modes share the same DTW selection and template steps.
# They differ only in what is submitted as the DTW input
#  and in how amplitude scaling is applied
# ============================================================

def forecast_realtime(sc_obs, pool_cycles, k):
    """
    Real-time mode forecast.

    The observed prefix is submitted as the DTW input.
    Amplitude scaling uses IDW-weighted analog parameters.
    An anchor correction Δ  connects the forecast to the
    last observed value.  The result is clipped at zero.

    Parameters
    ----------
    sc_obs      : observed SSN prefix (months 1…T)
    pool_cycles : parity-restricted candidate cycles
    k           : ensemble size

    Returns
    -------
    f          : forecast array (physical SSN)
    sel        : selected analog cycle numbers
    w          : IDW weights
    dist_list  : full DTW distance ranking
    target_len : length of forecast array
    """
    y_obs  = np.asarray(sc_obs, float)
    T      = len(y_obs)
    sel, w, dist_list = dtw_select(y_obs, pool_cycles, k)
    # Expected cycle length = mean of selected analog lengths
    target_len = max(T, int(round(
        np.mean([len(cycle_fits[c][1]) for c in sel]))))
    sh   = blend_shape_norm(sel, w, target_len)
    f    = apply_idw_scaling(sh, sel, w)
    # Anchor correction Δ = S_obs(T) - S_hat_raw(T)  (Eq. 8)
    delta = y_obs[-1] - f[T - 1]
    f    += delta
    f     = np.clip(f, 0.0, None)   # sunspot numbers are non-negative
    return f, sel, w, dist_list, target_len


def forecast_baseline(pool_cycles, k, target_len):
    """
    Baseline mode forecast.

    A smooth half-sinusoidal reference curve R(t) is
    submitted as the DTW input in place of any observations.
    Amplitude scaling uses pool climatology (median peak and min).

    Parameters
    ----------
    pool_cycles : parity-restricted candidate cycles
    k           : ensemble size
    target_len  : expected cycle length (mean of pool)

    Returns
    -------
    f, sel, w, dist_list
    """
    ref  = baseline_reference_curve(target_len)
    sel, w, dist_list = dtw_select(ref, pool_cycles, k)
    sh   = blend_shape_norm(sel, w, target_len)
    f    = apply_climatology_scaling(sh, pool_cycles)
    f    = np.clip(f, 0.0, None)
    return f, sel, w, dist_list


def cosine_blend(a, b, blend_len=6):
    """Six-month cosine-tapered join between two forecast arrays."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    L    = min(int(blend_len), len(a), len(b))
    if L <= 0:
        return np.concatenate([a, b])
    w   = 0.5 * (1 - np.cos(np.linspace(0, np.pi, L)))
    mid = (1 - w) * a[-L:] + w * b[:L]
    return np.concatenate([a[:-L], mid, b[L:]])


def _hindcast_rt(sc_obs, pool_cycles, k, full_len):
    """
    Hindcast variant of forecast_realtime with an explicit target length.

    Used ONLY for walk-forward validation and phase-residual
    bin construction.  The horizon is set to full_len
    (the actual length of the target cycle) so that validation MAE and
    residuals cover the complete cycle — including the declining tail
    of long cycles such as SC23 (149 months).

    The operational forecast_realtime uses the mean analog length as
    its horizon (appropriate when SC25's true end is unknown).
    Keeping these two functions separate avoids conflating operational
    forecasting logic with evaluation logic.

    Parameters
    ----------
    sc_obs      : observed prefix (months 1…T)
    pool_cycles : parity-restricted candidate cycles
    k           : ensemble size
    full_len    : target forecast horizon = len(cycle_fits[target][1])

    Returns
    -------
    f, sel, w, dist_list
    """
    y_obs = np.asarray(sc_obs, float)
    T     = len(y_obs)
    sel, w, dist_list = dtw_select(y_obs, pool_cycles, k)
    sh    = blend_shape_norm(sel, w, full_len)
    f     = apply_idw_scaling(sh, sel, w)
    delta = y_obs[-1] - f[T - 1]          # anchor correction
    f    += delta
    f     = np.clip(f, 0.0, None)          # non-negative SSN
    return f, sel, w, dist_list


print("Forecast builders defined  (including _hindcast_rt for validation).")

Forecast builders defined  (including _hindcast_rt for validation).


In [ ]:
# ============================================================
# §6  WALK-FORWARD VALIDATION — Solar Cycles 21–24
#
# Validation method:
#   • Target cycles: SC21, SC22, SC23, SC24
#   • SC20 excluded: the earliest target must still retain ≥1
#     same-parity candidate in its training pool.
#   • Candidates: only cycles strictly earlier than the target.
#   • Observation limit: first PREFIX_LEN=60 months only.
#   • k tested: {1, 2, 3, 4} for both modes.
# ============================================================

k_values = [1, 2, 3, 4]
VAL_CYCLES = [21, 22, 23, 24]

def _eval_rt(sc, k):
    """
    Real-time walk-forward error for one target cycle at given k.

    Uses _hindcast_rt with full_len=len(y) so that MAE is evaluated
    over the COMPLETE target cycle.
    (forecast_realtime would truncate to mean-analog length, which
    shortens the evaluation window for long cycles like SC23.)
    """
    _, y = cycle_fits[sc]
    pool = [c for c in range(13, sc) if c % 2 == sc % 2]
    m    = min(PREFIX_LEN, len(y))
    f, _, _, _ = _hindcast_rt(y[:m], pool, k, len(y))  # full cycle horizon
    n    = min(len(y), len(f))
    mae  = float(mean_absolute_error(y[:n], f[:n]))
    rmse = true_rmse(y[:n], f[:n])
    r2   = float(r2_score(y[:n], f[:n]))
    r    = float(pearsonr(y[:n], f[:n])[0])
    return mae, rmse, r2, r

def _eval_bl(sc, k):
    """Baseline walk-forward error for one target cycle at given k."""
    _, y = cycle_fits[sc]
    pool = [c for c in range(13, sc) if c % 2 == sc % 2]
    tlen = int(round(np.mean([len(cycle_fits[c][1]) for c in pool])))
    f, _, _, _ = forecast_baseline(pool, k, tlen)
    n    = min(len(y), len(f))
    mae  = float(mean_absolute_error(y[:n], f[:n]))
    rmse = true_rmse(y[:n], f[:n])
    r2   = float(r2_score(y[:n], f[:n]))
    r    = float(pearsonr(y[:n], f[:n])[0])
    return mae, rmse, r2, r

rt_rows, bl_rows = [], []
for k in k_values:
    rt_m = [_eval_rt(sc, k) for sc in VAL_CYCLES]
    bl_m = [_eval_bl(sc, k) for sc in VAL_CYCLES]
    rt_rows.append(("Real-time", k,
                    float(np.mean([m[0] for m in rt_m])),
                    float(np.mean([m[1] for m in rt_m])),
                    float(np.mean([m[2] for m in rt_m])),
                    float(np.mean([m[3] for m in rt_m]))))
    bl_rows.append(("Baseline", k,
                    float(np.mean([m[0] for m in bl_m])),
                    float(np.mean([m[1] for m in bl_m])),
                    float(np.mean([m[2] for m in bl_m])),
                    float(np.mean([m[3] for m in bl_m]))))

cols = ["Mode", "k", "MAE", "RMSE", "R2", "r"]
val_df    = pd.DataFrame(rt_rows, columns=cols)
bl_val_df = pd.DataFrame(bl_rows, columns=cols)

# k selection: lowest MAE in each mode
BEST_K_RT = int(val_df.loc[val_df.MAE.idxmin(), "k"])
BEST_K_BL = int(bl_val_df.loc[bl_val_df.MAE.idxmin(), "k"])

# Manuscript Table 1
table1 = pd.concat([val_df, bl_val_df]).sort_values(["Mode","k"],
             ascending=[False,True]).reset_index(drop=True)
table1.to_csv(os.path.join(OUT_TAB, "table1_validation.csv"), index=False)

print("Manuscript Table 1 — Walk-forward validation (SC21–24):")
print(table1.round(2).to_string(index=False))
print(f"\n  BEST_K_RT = {BEST_K_RT}  (manuscript: 3)")
print(f"  BEST_K_BL = {BEST_K_BL}  (manuscript: 2)")

Manuscript Table 1 — Walk-forward validation (SC21–24):
     Mode  k   MAE  RMSE   R2    r
Real-time  1 29.93 36.39 0.64 0.90
Real-time  2 25.99 32.27 0.73 0.94
Real-time  3 19.40 25.00 0.83 0.96
Real-time  4 21.35 26.64 0.81 0.96
 Baseline  1 27.23 33.43 0.62 0.95
 Baseline  2 26.39 31.81 0.65 0.96
 Baseline  3 27.54 32.62 0.63 0.96
 Baseline  4 27.23 32.85 0.62 0.96

  BEST_K_RT = 3  (manuscript: 3)
  BEST_K_BL = 2  (manuscript: 2)


In [ ]:
# ============================================================
# §7  PHASE-AWARE RESIDUAL BOOTSTRAP
#
#
# Step 1: Collect walk-forward hindcast residuals for every cycle
#   in the analog pool, sorted into BINS=12 equal phase bins.
#   Each bin is mean-centred so resampled errors carry no net bias.
#
# Step 2: For a new forecast, draw N_BOOT=5000 residuals from the
#   matching phase bin and add to the point forecast.  Residuals
#   are amplitude-scaled by ramp(t) = max(ν, sqrt(f(t)/f_max)),
#   which reflects the Poisson-like relationship between sunspot
#   variability and activity level.  ν=0.35 prevents
#   unrealistic collapse near solar minimum.
#
# Step 3: Conformal calibration — find scale factors s80, s98
#   so that the intervals achieve target coverage on SC21–24.
# ============================================================

def phase_bin_indices(n, bins=BINS):
    """Map n time steps to BINS equal phase bins."""
    idx = (np.linspace(0, 1, n, endpoint=True) * bins).astype(int)
    return np.minimum(idx, bins - 1)


def build_phase_residual_bins(pool_cycles, k_val, prefix_len=PREFIX_LEN):
    """
    Collect walk-forward residuals from all cycles in pool_cycles,
    using only earlier same-parity cycles as candidates.
    Returns BINS mean-centred residual lists.
    """
    bins_raw = [[] for _ in range(BINS)]
    for cyc in pool_cycles:
        train = [c for c in range(13, cyc) if c % 2 == cyc % 2]
        if not train:
            continue
        _, y = cycle_fits[cyc]
        m    = min(prefix_len, len(y))
        # Use _hindcast_rt with the FULL cycle length so that residuals
        # cover the complete cycle — including the declining tail of
        # long cycles (e.g. SC23: 149 months).  This is essential for
        # the late-phase bins (bins 9–12) to be adequately populated.
        f, _, _, _ = _hindcast_rt(y[:m], train,
                                   min(k_val, len(train)), len(y))
        n_min = min(len(y), len(f))
        r     = y[:n_min] - f[:n_min]
        idx   = phase_bin_indices(n_min, bins=BINS)
        for t in range(n_min):
            bins_raw[idx[t]].append(float(r[t]))
    # Mean-centre each bin to remove bias
    return [
        [x - float(np.mean(b)) for x in b] if b else []
        for b in bins_raw
    ]


def bootstrap_pi(forecast, bins_zero, draws=N_BOOT,
                 q=(0.10, 0.90), seed=0, nu=NU_PUB):
    """
    Phase-aware residual bootstrap prediction interval.

    Parameters
    ----------
    forecast   : point forecast array
    bins_zero  : mean-centred residual bins from build_phase_residual_bins
    draws      : bootstrap sample size (default N_BOOT=5000)
    q          : (lower, upper) quantile pair
    seed       : RNG seed for reproducibility
    nu         : variance floor — prevents PI collapsing to zero near
                 solar minimum ( default NU_PUB=0.35)

    Returns
    -------
    lo, hi : lower and upper PI bounds (clipped at zero)
    """
    rng   = np.random.default_rng(seed)
    f     = np.asarray(forecast, float)
    N     = len(f)
    idx   = phase_bin_indices(N, bins=len(bins_zero))
    f_pos = np.clip(f, 0.0, None)
    f_max = max(float(np.max(f_pos)), 1.0)
    # Amplitude scaling ramp
    ramp  = np.clip(np.sqrt(f_pos / f_max), nu, 1.0)
    lo, hi = np.empty(N), np.empty(N)
    for t in range(N):
        pool = np.asarray(bins_zero[idx[t]], float)
        if pool.size == 0:
            lo[t] = hi[t] = f[t]
        else:
            samp  = rng.choice(pool, size=draws, replace=True) * ramp[t] + f[t]
            lo[t] = max(0.0, float(np.quantile(samp, q[0])))
            hi[t] = float(np.quantile(samp, q[1]))
    return lo, hi


print("Building phase-residual bins from real walk-forward hindcasts …")
print("  (odd pool — used for SC25) …")
odd_bins0  = build_phase_residual_bins(odd_pool,  BEST_K_RT)
print("  (even pool — used for SC26) …")
even_bins0 = build_phase_residual_bins(even_pool, BEST_K_BL)
print("  Done.")



Building phase-residual bins from real walk-forward hindcasts …
  (odd pool — used for SC25) …
  (even pool — used for SC26) …
  Done.


In [ ]:
# ============================================================
# §7.1  CONFORMAL CALIBRATION
#
# Find scale factors s80 and s98 so that the bootstrapped
# intervals achieve empirical coverage ≥80% and ≥98% on the
# four held-out validation cycles SC21–SC24.
# ============================================================

# ── Parity-specific conformal calibration ─────────────────────────────────
# The calibration scale factors s80 and s98 are found using only the two
# odd-cycle held-out validation targets: SC21 and SC23.
#
# Scientific justification :
#   • The operational target is SC25 (odd). Odd and even cycles have
#     demonstrably different residual distributions : odd
#     cycles are broader, more asymmetric, and heavier-tailed.
#   • Calibrating the odd-cycle error model (odd_bins0) against odd-cycle
#     validation data (SC21, SC23) is internally consistent: we ask how
#     well odd bins cover odd-cycle errors, which is exactly what we need.
#   • Mixing even-cycle validation targets (SC22, SC24) into the
#     calibration would dilute the estimate with a narrower error
#     distribution, producing a scale factor s80 that undershoots the
#     coverage target for the primary odd-cycle target (SC25).
#   • This parity-matched design is consistent with the parity-restricted
#     analog pools used throughout the framework.
#
# For SC26 (even), the same s80 is applied to even_bins (even_bins_80),
# producing prediction intervals calibrated to even-cycle residual scale.
# ──────────────────────────────────────────────────────────────────────────
ODD_VAL_CYCLES = [21, 23]   # odd-cycle held-out targets for calibration

def _val_coverage(scale, target_q, cycles=ODD_VAL_CYCLES):
    """
    Empirical coverage of s-scaled odd-cycle bootstrap PI on validation
    cycles.  Defaults to odd held-out cycles (SC21, SC23) so that
    calibration is parity-matched to the primary target (SC25, odd).
    """
    bs   = [[scale * x for x in b] for b in odd_bins0]
    covs = []
    for sc in cycles:
        _, y = cycle_fits[sc]
        pool = [c for c in range(13, sc) if c % 2 == sc % 2]
        if not pool:
            continue
        m = min(PREFIX_LEN, len(y))
        f, _, _, _, _ = forecast_realtime(y[:m], pool,
                                           min(BEST_K_RT, len(pool)))
        n_min = min(len(y), len(f))
        lo, hi = bootstrap_pi(f[:n_min], bs, q=target_q,
                               seed=RNG_SEED + sc)
        covs.append(float(np.mean((y[:n_min] >= lo) & (y[:n_min] <= hi))))
    return float(np.mean(covs))

# Grid search — coarse then fine
grid_coarse = np.linspace(0.5, 3.0, 51)
s80 = float(min(grid_coarse,
               key=lambda s: abs(_val_coverage(s, (0.10, 0.90)) - 0.80)))
s98 = float(min(grid_coarse,
               key=lambda s: abs(_val_coverage(s, (0.01, 0.99)) - 0.98)))

# Apply scale factors to produce calibrated bins
odd_bins_80  = [[s80 * x for x in b] for b in odd_bins0]
odd_bins_98  = [[s98 * x for x in b] for b in odd_bins0]
even_bins_80 = [[s80 * x for x in b] for b in even_bins0]
even_bins_98 = [[s98 * x for x in b] for b in even_bins0]

cov80_val = _val_coverage(s80, (0.10, 0.90))
cov98_val = _val_coverage(s98, (0.01, 0.99))

print("Conformal calibration — parity-matched (Section 3.9):")
print(f"  Calibration cycles : {ODD_VAL_CYCLES}  (odd-cycle held-out targets)")
print(f"  Residual pool used : odd_bins0  (matches target parity)")
print(f"  s80 = {s80:.2f}  →  val 80% coverage (SC21,SC23) = {cov80_val:.3f}")
print(f"  s98 = {s98:.2f}  →  val 98% coverage (SC21,SC23) = {cov98_val:.3f}")
print("  Design: calibrate odd error model on odd validation data only.")
print("  For SC26, s80/s98 are applied to even_bins — parity-scaled.")

for cal_cycle in ODD_VAL_CYCLES:
    s_loo = float(min(grid_coarse,
        key=lambda s: abs(_val_coverage(s, (0.10, 0.90),
                          cycles=[cal_cycle]) - 0.80)))
    print(f"  LOO s80 (calibrated on SC{cal_cycle} only): {s_loo:.2f}")

Conformal calibration — parity-matched (Section 3.9):
  Calibration cycles : [21, 23]  (odd-cycle held-out targets)
  Residual pool used : odd_bins0  (matches target parity)
  s80 = 1.40  →  val 80% coverage (SC21,SC23) = 0.809
  s98 = 2.10  →  val 98% coverage (SC21,SC23) = 0.980
  Design: calibrate odd error model on odd validation data only.
  For SC26, s80/s98 are applied to even_bins — parity-scaled.
  LOO s80 (calibrated on SC21 only): 0.95
  LOO s80 (calibrated on SC23 only): 1.85


In [ ]:
# ============================================================
# §8  OPERATIONAL FORECASTS
# ============================================================

# --- SC25 real-time forecast -------------------
f25_rt, sel25_rt, w25_rt, dist25_rt, L25 = forecast_realtime(
    y_obs_25, odd_pool, BEST_K_RT)

lo80_rt, hi80_rt = bootstrap_pi(f25_rt, odd_bins_80,
                                 q=(0.10,0.90), seed=RNG_SEED+80)
lo98_rt, hi98_rt = bootstrap_pi(f25_rt, odd_bins_98,
                                 q=(0.01,0.99), seed=RNG_SEED+98)

pk25_rt = int(np.argmax(f25_rt))
cov80_rt = float(np.mean((y_obs_25 >= lo80_rt[:t_len]) &
                          (y_obs_25 <= hi80_rt[:t_len])))
cov98_rt = float(np.mean((y_obs_25 >= lo98_rt[:t_len]) &
                          (y_obs_25 <= hi98_rt[:t_len])))

obs_in_rt = int(np.sum((y_obs_25 >= lo80_rt[:t_len]) &
                        (y_obs_25 <= hi80_rt[:t_len])))
obs_in_rt98 = int(np.sum((y_obs_25 >= lo98_rt[:t_len]) &
                          (y_obs_25 <= hi98_rt[:t_len])))

mae25_rt   = float(mean_absolute_error(y_obs_25, f25_rt[:t_len]))
rmse25_rt  = true_rmse(y_obs_25, f25_rt[:t_len])
r2_25_rt   = float(r2_score(y_obs_25, f25_rt[:t_len]))
r_25_rt    = float(pearsonr(y_obs_25, f25_rt[:t_len])[0])

print("SC25 Real-Time Forecast :")
print(f"  k        = {BEST_K_RT}  ")
print(f"  Analogs  = {sel25_rt}  ")
print(f"  Peak SSN = {np.max(f25_rt):.1f} at month {pk25_rt}  ")
print(f"  MAE      = {mae25_rt:.2f}  RMSE = {rmse25_rt:.2f}  R²={r2_25_rt:.3f}")
print(f"  80% PI coverage: {obs_in_rt}/{t_len} = {100*cov80_rt:.1f}%  ")
print(f"  98% PI coverage: {obs_in_rt98}/{t_len} = {100*cov98_rt:.1f}%  ")
print(f"  80% PI at peak: [{lo80_rt[pk25_rt]:.1f}, {hi80_rt[pk25_rt]:.1f}]  "
      f"width={hi80_rt[pk25_rt]-lo80_rt[pk25_rt]:.1f}")

print()

# --- SC25 baseline forecast --------------------
mean_len_odd = int(round(np.mean([len(cycle_fits[c][1]) for c in odd_pool])))
f25_bl, sel25_bl, w25_bl, dist25_bl = forecast_baseline(
    odd_pool, BEST_K_BL, mean_len_odd)
lo80_bl25, hi80_bl25 = bootstrap_pi(f25_bl, odd_bins_80,
                                     q=(0.10,0.90), seed=RNG_SEED+81)
lo98_bl25, hi98_bl25 = bootstrap_pi(f25_bl, odd_bins_98,
                                     q=(0.01,0.99), seed=RNG_SEED+250)
obs_in_bl80 = int(np.sum((y_obs_25 >= lo80_bl25[:t_len]) &
                          (y_obs_25 <= hi80_bl25[:t_len])))
cov80_bl = float(np.mean((y_obs_25 >= lo80_bl25[:t_len]) &
                          (y_obs_25 <= hi80_bl25[:t_len])))

print(f"SC25 Baseline Forecast: analogs={sel25_bl}  "
      f"peak={np.max(f25_bl):.1f}  ")
print(f"  Baseline 80% PI coverage: {obs_in_bl80}/{t_len} = {100*cov80_bl:.1f}%  ")

print()

# --- SC26 baseline forecast ----------
mean_len_even = int(round(np.mean([len(cycle_fits[c][1]) for c in even_pool])))
f26_bl, sel26_bl, w26_bl, dist26_bl = forecast_baseline(
    even_pool, BEST_K_BL, mean_len_even)
lo80_bl26, hi80_bl26 = bootstrap_pi(f26_bl, even_bins_80,
                                     q=(0.10,0.90), seed=RNG_SEED+82)
lo98_bl26, hi98_bl26 = bootstrap_pi(f26_bl, even_bins_98,
                                     q=(0.01,0.99), seed=RNG_SEED+260)
pk26 = int(np.argmax(f26_bl))

print(f"SC26 Baseline Forecast: analogs={sel26_bl}  "
      f"peak={np.max(f26_bl):.1f}  ")
print(f"  80% PI at peak: [{lo80_bl26[pk26]:.1f}, {hi80_bl26[pk26]:.1f}]")
print(f"  98% PI at peak: [{lo98_bl26[pk26]:.1f}, {hi98_bl26[pk26]:.1f}]  ")

# --- Two-cycle stitched projection  ------
f25_26_stitched = cosine_blend(f25_rt, f26_bl, blend_len=6)

SC25 Real-Time Forecast :
  k        = 3  
  Analogs  = [17, 15, 21]  
  Peak SSN = 173.4 at month 48  
  MAE      = 12.28  RMSE = 16.37  R²=0.908
  80% PI coverage: 53/60 = 88.3%  
  98% PI coverage: 60/60 = 100.0%  
  80% PI at peak: [141.5, 207.1]  width=65.5

SC25 Baseline Forecast: analogs=[19, 21]  peak=188.7  
  Baseline 80% PI coverage: 47/60 = 78.3%  

SC26 Baseline Forecast: analogs=[18, 22]  peak=143.3  
  80% PI at peak: [119.2, 173.2]
  98% PI at peak: [83.5, 210.5]  


In [ ]:
# ============================================================
# FIGURE 1 — Full smoothed SSN record with cycle boundaries
# ============================================================
def make_fig1():
    fig, ax = plt.subplots(figsize=(11.5, 3.8))
    ax.plot(df.index, df["SmoothedSSN"].values,
            linewidth=0.9, color="#01579B")
    for sc, (st, en) in cycle_boundaries.items():
        st2 = pd.to_datetime(st)
        en2 = pd.to_datetime(en)
        ax.axvline(st2, linewidth=0.55, color="#78909C")
        mid = st2 + (en2 - st2) / 2
        ax.text(mid, ax.get_ylim()[1] * 0.91,
                f"SC{sc}", ha="center", va="top", fontsize=7.5)
    ax.axvline(sc25_start, linestyle="--",
               linewidth=1.0, color="tab:orange")
    ax.set_ylim(0)
    ax.set_title(
        "Monthly Smoothed Sunspot Number (SILSO v2.0) "
        "with Solar Cycle Labels")
    ax.set_ylabel("Smoothed SSN")
    ax.set_xlabel("Year")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    ax.margins(x=0.01)
    return fig

safe_save(make_fig1(), "fig01_full_series_cycle_labels")

  saved: fig01_full_series_cycle_labels.pdf / .eps


In [ ]:
# ============================================================
#  Time- and amplitude-normalised cycle profiles
# ============================================================
def make_fig2():
    cycles = list(range(13, 25))
    fig, axes = plt.subplots(3, 4, figsize=(11.5, 7),
                              sharex=True, sharey=True)
    for i, sc in enumerate(cycles):
        ax = axes.ravel()[i]
        _, y = cycle_fits[sc]
        ax.plot(np.linspace(0, 1, len(y)), _norm01(y),
                linewidth=1.6, color="#01579B")
        ax.set_title(f"SC{sc}", fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.subplots_adjust(left=0.08, right=0.98,
                         bottom=0.12, top=0.92,
                         hspace=0.35, wspace=0.25)
    fig.suptitle(
        "Time- and Amplitude-Normalized Solar Cycles (SC13–SC24)",
        fontsize=12)
    fig.supxlabel("Normalized Time (τ)", fontsize=11, y=0.04)
    fig.supylabel("Normalized SSN (0–1)", fontsize=11, x=0.02)
    fig.text(0.01, 0.01, STAMP, fontsize=9)
    return fig

safe_save(make_fig2(), "fig02_normalized_cycles_grid")

  saved: fig02_normalized_cycles_grid.pdf / .eps


In [ ]:
# ============================================================
# FIGURES 4 & 5 — DTW distance bar charts
# SC25 real-time observed prefix vs. odd-cycle pool
#  Baseline reference curve vs. even-cycle pool (SC26)
# ============================================================
def make_dtw_dist_fig(dist_list, pool, title, stem):
    pool_set = set(pool)
    rows  = [(c, d) for c, d in dist_list if c in pool_set]
    rows.sort(key=lambda x: x[1])   # sorted ascending
    labels = [f"{c}" for c, _ in rows]
    dists  = [d for _, d in rows]
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    colors = ["#01579B" if d == min(dists[:3]) or
                           dists.index(d) < (3 if len(pool)==6 else 2)
              else "steelblue" for d in dists]
    # Highlight selected analogs (lowest k distances)
    k_used = BEST_K_RT if "Observed" in title else BEST_K_BL
    bar_colors = ["#E64A19" if i < k_used else "steelblue"
                  for i in range(len(rows))]
    ax.bar(labels, dists, color=bar_colors, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Candidate solar cycle")
    ax.set_ylabel("DTW distance (normalised units)")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    safe_save(fig, stem)

make_dtw_dist_fig(
    dist25_rt, odd_pool,
    "DTW Distances: SC25 Observed Prefix vs. Odd-Cycle Analogs",
    "fig04_dtw_distances_sc25_rt")

make_dtw_dist_fig(
    dist26_bl, even_pool,
    "DTW Distances: Baseline Reference Shape vs. Even-Cycle Analogs",
    "fig05_dtw_distances_sc26_baseline")

  saved: fig04_dtw_distances_sc25_rt.pdf / .eps
  saved: fig05_dtw_distances_sc26_baseline.pdf / .eps


In [ ]:
# ============================================================
# Walk-forward residual boxplots by phase bin
# Boxplots (not histograms) because they show median, IQR,
# whiskers, and individual outliers simultaneously
# ============================================================
def make_residual_boxplot_fig(bins_zero, title, stem):
    fig, axes = plt.subplots(3, 4, figsize=(11.5, 6.5),
                              constrained_layout=True)
    rng_jitter = np.random.default_rng(99)
    for i, ax in enumerate(axes.ravel()):
        data = np.asarray(bins_zero[i], float)
        if data.size > 0:
            ax.boxplot(
                data, vert=True, patch_artist=True, widths=0.5,
                boxprops=dict(facecolor="#B2DFDB", edgecolor="#00796B",
                              linewidth=1.2),
                medianprops=dict(color="#E64A19", linewidth=2.0),
                whiskerprops=dict(color="#00796B", linewidth=1.0,
                                  linestyle="--"),
                capprops=dict(color="#00796B", linewidth=1.5),
                flierprops=dict(marker="o", markerfacecolor="#01579B",
                                markeredgecolor="none",
                                markersize=3.5, alpha=0.65),
            )
            jitter = rng_jitter.uniform(-0.14, 0.14, size=len(data))
            ax.scatter(np.ones(len(data)) + jitter, data,
                       color="#01579B", alpha=0.22, s=7, zorder=2)
        ax.axhline(0.0, color="black", linewidth=0.85, linestyle=":")
        ax.set_title(f"Bin {i+1}", fontsize=9.5)
        ax.set_xticks([])
        ax.set_ylabel("Residual (SSN)" if i % 4 == 0 else "")
    fig.suptitle(title, fontsize=11)
    fig.text(0.01, 0.01, STAMP, fontsize=9)
    safe_save(fig, stem)

make_residual_boxplot_fig(
    odd_bins_98,
    "Walk-Forward Residuals by Phase Bin — Odd Cycles\n"
    "(box: IQR; orange line: median; dashed whiskers: 1.5×IQR; "
    "dots: outliers and data points)",
    "fig06_residuals_odd_cycles")

make_residual_boxplot_fig(
    even_bins_98,
    "Walk-Forward Residuals by Phase Bin — Even Cycles\n"
    "(box: IQR; orange line: median; dashed whiskers: 1.5×IQR; "
    "dots: outliers and data points)",
    "fig07_residuals_even_cycles")

  saved: fig06_residuals_odd_cycles.pdf / .eps


  saved: fig07_residuals_even_cycles.pdf / .eps


In [ ]:
# ============================================================
# FIGURE 8 — Morphological comparison: SC25 prefix vs. analogs
# SC25 is shown ONLY up to the observation horizon τ ≈ 0.48
# (month 60 / mean analog length), not beyond it.
# ============================================================
def make_fig8():
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    mean_L  = np.mean([len(cycle_fits[c][1]) for c in sel25_rt])
    tau_cut = t_len / mean_L

    color_list = ["#1565C0", "#2E7D32", "#6A1B9A"]
    for i, sc in enumerate(sel25_rt):
        _, y = cycle_fits[sc]
        L    = len(y)
        ax.plot(np.linspace(0, 1, L), _norm01(y),
                color=color_list[i], linewidth=2.0,
                linestyle="--",
                label=f"SC{sc}  (analog, {L} months)", zorder=3)
        tau_60  = t_len / L
        idx_60  = min(int(tau_60 * (L - 1)), L - 1)
        ax.plot(tau_60, _norm01(y)[idx_60], "o",
                color=color_list[i], markersize=6,
                markeredgecolor="white", markeredgewidth=1.2,
                zorder=6)

    sc25_tau = np.linspace(0, tau_cut, t_len)
    ax.plot(sc25_tau, _norm01(y_obs_25),
            color="black", linewidth=2.6,
            label=f"SC25 observed prefix ({t_len} months)", zorder=5)
    ax.axvline(tau_cut, color="black", linewidth=1.1,
               linestyle=":", alpha=0.75, zorder=4)
    ax.axvspan(tau_cut, 1.02, alpha=0.05, color="gray", zorder=1)
    ax.text((tau_cut + 1.01) / 2, 0.40,
            "Unobserved\n(forecast region)",
            fontsize=9, ha="center", va="center",
            color="gray", style="italic")

    dot_proxy = mlines.Line2D(
        [], [], marker="o", color="gray",
        markerfacecolor="gray", markersize=6,
        markeredgecolor="white", markeredgewidth=1.2,
        linestyle="none", label="Analog position at month 60")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles + [dot_proxy],
              labels + ["Analog position at month 60"],
              frameon=True, framealpha=0.90,
              fontsize=8.5, loc="upper left",
              borderpad=0.8, labelspacing=0.45)

    ax.set_xlim(-0.01, 1.03)
    ax.set_ylim(-0.05, 1.22)
    ax.set_xlabel("Normalised time  (τ = month / cycle length)")
    ax.set_ylabel("Normalised amplitude  (0 = min, 1 = max)")
    ax.set_title(
        "Morphological comparison: SC25 observed prefix vs. "
        "real-time analogs\n"
        f"SC25 shown only up to observation horizon "
        f"(τ ≈ {tau_cut:.2f})")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(make_fig8(), "fig08_sc25_normalized_analogs")

  saved: fig08_sc25_normalized_analogs.pdf / .eps


In [ ]:
# ============================================================
# Real-time SC25 forecast with 80% and 98% PI
# ============================================================
def make_dual_pi_fig(f, lo80, hi80, lo98, hi98,
                     title, stem, obs=None, obs_len=None):
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    x = np.arange(len(f))
    # 98% PI — light, secondary
    ax.fill_between(x, lo98, hi98,
                    facecolor="#F5F5F5", alpha=0.70, zorder=1)
    ax.plot(x, lo98, color="#9E9E9E", linewidth=0.85,
            linestyle="--", zorder=2)
    ax.plot(x, hi98, color="#9E9E9E", linewidth=0.85,
            linestyle="--", zorder=2)
    # 80% PI — primary
    ax.fill_between(x, lo80, hi80,
                    facecolor="#B2DFDB", edgecolor="#00796B",
                    linewidth=0.7, alpha=0.90, zorder=3)
    ax.plot(x, f, color="#01579B", linewidth=2.2, zorder=5,
            label="Forecast")
    if obs is not None:
        oxl = obs_len if obs_len else len(obs)
        ax.plot(np.arange(oxl), obs[:oxl],
                color="black", linewidth=1.8, zorder=6,
                label="Observed")
    p80 = mpatches.Patch(facecolor="#B2DFDB", edgecolor="#00796B",
                          linewidth=0.7, label="80% PI  (primary)")
    p98 = mpatches.Patch(facecolor="#F5F5F5", edgecolor="#9E9E9E",
                          linestyle="--", label="98% PI")
    handles = ([mlines.Line2D([],[],color="black",linewidth=1.8,
                               label="Observed")]
               if obs is not None else [])
    handles += [mlines.Line2D([],[],color="#01579B",linewidth=2.2,
                               label="Forecast"), p80, p98]
    ax.legend(handles=handles, frameon=True, framealpha=0.92,
              fontsize=9, loc="upper right", borderpad=0.8)
    ax.set_xlabel("Months since cycle start")
    ax.set_ylabel("Smoothed SSN")
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    safe_save(fig, stem)

make_dual_pi_fig(
    f25_rt, lo80_rt, hi80_rt, lo98_rt, hi98_rt,
    title=f"Real-time SC25 forecast (k={BEST_K_RT}, "
          f"{t_len} months obs)",
    stem="fig09_sc25_realtime_forecast_pi",
    obs=y_obs_25, obs_len=t_len)

  saved: fig09_sc25_realtime_forecast_pi.pdf / .eps


In [ ]:
# ============================================================
#Baseline vs. real-time SC25 (dual panel)
# Left : full forecast window
# Right: zoom on observed window with coverage check
# ============================================================
def make_fig10():
    pk_rt = int(np.argmax(f25_rt))
    pk_bl = int(np.argmax(f25_bl))
    PI_w_rt = float(hi80_rt[pk_rt]    - lo80_rt[pk_rt])
    PI_w_bl = float(hi80_bl25[pk_bl]  - lo80_bl25[pk_bl])
    obs_in_bl = int(np.sum((y_obs_25 >= lo80_bl25[:t_len]) &
                            (y_obs_25 <= hi80_bl25[:t_len])))

    fig = plt.figure(figsize=(14.0, 5.2))
    gs  = fig.add_gridspec(
        1, 2, width_ratios=[2.2, 1], wspace=0.28,
        left=0.06, right=0.98, top=0.88, bottom=0.12)
    ax  = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    x_rt = np.arange(len(f25_rt))
    x_bl = np.arange(len(f25_bl))

    ax.fill_between(x_bl, lo80_bl25, hi80_bl25,
                    facecolor="#FFCCBC", edgecolor="#E64A19",
                    linewidth=0.5, alpha=0.55, zorder=1)
    ax.fill_between(x_rt, lo80_rt,   hi80_rt,
                    facecolor="#B2DFDB", edgecolor="#00796B",
                    linewidth=0.6, alpha=0.82, zorder=2)
    ax.plot(x_bl, f25_bl, color="#E64A19",
            linewidth=1.8, linestyle="--", zorder=4)
    ax.plot(x_rt, f25_rt, color="#01579B",
            linewidth=2.2, zorder=5)
    ax.plot(np.arange(t_len), y_obs_25,
            color="black", linewidth=2.0, zorder=6)
    ax.axvline(t_len - 1, color="black", linewidth=1.0,
               linestyle=":", alpha=0.55, zorder=3)

    ax.annotate(
        f"RT peak: {np.max(f25_rt):.0f} SSN",
        xy=(pk_rt, np.max(f25_rt)),
        xytext=(pk_rt + 8, np.max(f25_rt) - 30),
        fontsize=8.5, color="#01579B",
        arrowprops=dict(arrowstyle="->", color="#01579B", lw=0.9),
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                  edgecolor="#01579B", alpha=0.85))
    ax.annotate(
        f"Baseline peak: {np.max(f25_bl):.0f} SSN",
        xy=(pk_bl, np.max(f25_bl)),
        xytext=(max(pk_bl - 38, 2), np.max(f25_bl) + 15),
        fontsize=8.5, color="#E64A19",
        arrowprops=dict(arrowstyle="->", color="#E64A19", lw=0.9),
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                  edgecolor="#E64A19", alpha=0.85))

    # Legend
    p_bl = mpatches.Patch(
        facecolor="#FFCCBC", edgecolor="#E64A19", linewidth=0.5,
        alpha=0.7,
        label=f"80% PI — baseline  (peak width: {PI_w_bl:.0f} SSN)")
    p_rt = mpatches.Patch(
        facecolor="#B2DFDB", edgecolor="#00796B", linewidth=0.6,
        alpha=0.9,
        label=f"80% PI — real-time  (peak width: {PI_w_rt:.0f} SSN)")
    l_bl = mlines.Line2D([],[],color="#E64A19",linewidth=1.8,
                          linestyle="--",
                          label=f"Baseline  "
                                f"(SC{sel25_bl[0]} & SC{sel25_bl[1]})")
    l_rt = mlines.Line2D([],[],color="#01579B",linewidth=2.2,
                          label=f"Real-time  "
                                f"(SC{sel25_rt[0]}·SC{sel25_rt[1]}"
                                f"·SC{sel25_rt[2]})")
    l_obs = mlines.Line2D([],[],color="black",linewidth=2.0,
                           label="Observed SC25  (Jan 2020 – Dec 2024)")
    ax.legend(handles=[l_obs, l_rt, p_rt, l_bl, p_bl],
              frameon=True, framealpha=0.94, fontsize=8.5,
              loc="lower left", borderpad=0.8, labelspacing=0.4)
    y_max = max(np.max(hi80_bl25), np.max(hi80_rt),
                np.max(y_obs_25)) * 1.08
    ax.set_xlim(-2, max(len(f25_rt), len(f25_bl)) + 4)
    ax.set_ylim(0, y_max)
    ax.set_xlabel("Months since SC25 start")
    ax.set_ylabel("Smoothed SSN")
    ax.set_title(
        "SC25: baseline vs. real-time forecast with 80% prediction intervals\n"
        f"Observations narrow the PI and shift peak from "
        f"{np.max(f25_bl):.0f} to {np.max(f25_rt):.0f} SSN",
        fontsize=10)
    ax.text(0.01, 0.47, STAMP, transform=ax.transAxes, fontsize=9)

    # Right panel — observed window zoom
    xo = np.arange(t_len)
    ax2.fill_between(xo, lo80_bl25[:t_len], hi80_bl25[:t_len],
                     facecolor="#FFCCBC", edgecolor="#E64A19",
                     linewidth=0.5, alpha=0.55, zorder=1)
    ax2.fill_between(xo, lo80_rt[:t_len], hi80_rt[:t_len],
                     facecolor="#B2DFDB", edgecolor="#00796B",
                     linewidth=0.6, alpha=0.82, zorder=2)
    ax2.plot(xo, f25_bl[:t_len], color="#E64A19",
             linewidth=1.8, linestyle="--", zorder=4)
    ax2.plot(xo, f25_rt[:t_len], color="#01579B",
             linewidth=2.2, zorder=5)
    ax2.plot(xo, y_obs_25, color="black",
             linewidth=2.0, zorder=6)
    ax2.text(
        0.98, 0.04,
        f"RT 80% PI: {obs_in_rt}/{t_len} ({100*cov80_rt:.0f}%) ✓\n"
        f"BL 80% PI: {obs_in_bl}/{t_len} ({100*cov80_bl:.0f}%)",
        transform=ax2.transAxes, fontsize=8.5,
        va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                  edgecolor="#BDBDBD", alpha=0.95))
    for ls, col, lbl in [
            ("--","#E64A19","Baseline"),
            ("-", "#01579B","Real-time"),
            ("-", "black",  "Observed")]:
        ax2.plot([],[],color=col,linewidth=1.8,linestyle=ls,label=lbl)
    ax2.legend(frameon=True, framealpha=0.92, fontsize=8.5,
               loc="upper left", borderpad=0.6, labelspacing=0.3)
    y_max2 = max(np.max(hi80_bl25[:t_len]),
                 np.max(hi80_rt[:t_len]),
                 np.max(y_obs_25)) * 1.10
    ax2.set_xlim(0, t_len - 1); ax2.set_ylim(0, y_max2)
    ax2.set_xlabel("Months since SC25 start")
    ax2.set_ylabel("Smoothed SSN")
    ax2.set_title(
        f"Observed window (months 0–{t_len-1})\n"
        "80% PI coverage check", fontsize=10)
    return fig

safe_save(make_fig10(), "fig10_sc25_baseline_vs_realtime")

  saved: fig10_sc25_baseline_vs_realtime.pdf / .eps


In [ ]:
# ============================================================
# Retrospective forecast evolution (Table 3)
# ============================================================
cutoffs     = ["2020-12-01","2021-12-01","2022-12-01",
               "2023-12-01","2024-12-01"]
evol_data   = {}
evol_rows   = []

# Baseline row
ap_peak_dt = sc25_start + pd.DateOffset(
    months=int(np.argmax(f25_bl)))
evol_rows.append({
    "Cutoff":"Pre-SC25","Months":0,
    "Mode":f"Ref. shape (k={BEST_K_BL})",
    "Analogs":", ".join(map(str, sel25_bl)),
    "Peak SSN":round(float(np.max(f25_bl)), 1),
    "Peak Date":ap_peak_dt.strftime("%Y-%m"),
})

for c_date in cutoffs:
    sim_series = df.loc[sc25_start:pd.to_datetime(c_date),
                        "SmoothedSSN"].dropna()
    y_sim = sim_series.values.astype(float)
    f_sim, sel_sim, _, _, _ = forecast_realtime(
        y_sim, odd_pool, BEST_K_RT)
    evol_data[c_date] = {"y_obs": y_sim, "f_rt": f_sim}
    evol_rows.append({
        "Cutoff":c_date,"Months":len(y_sim),
        "Mode":f"Obs. prefix (k={BEST_K_RT})",
        "Analogs":", ".join(map(str, sel_sim)),
        "Peak SSN":round(float(np.max(f_sim)), 1),
        "Peak Date":(sc25_start + pd.DateOffset(
            months=int(np.argmax(f_sim)))).strftime("%Y-%m"),
    })

evol_df = pd.DataFrame(evol_rows)
evol_df.to_csv(os.path.join(OUT_TAB, "table3_sc25_evolution.csv"),
               index=False)
print("Manuscript Table 3:")
print(evol_df.to_string(index=False))

def make_fig11():
    fig, ax = plt.subplots(figsize=(10.0, 5.5))
    ax.plot(np.arange(t_len), y_obs_25,
            color="black", linewidth=2.2,
            label="Observed SC25 (to Dec 2024)", zorder=5)
    ax.plot(np.arange(len(f25_bl)), f25_bl,
            color="gray", linestyle=":", linewidth=2.0,
            label=f"Baseline (k={BEST_K_BL})", zorder=2)
    colors_e = cm.plasma(np.linspace(0.1, 0.8, len(evol_data)))
    for i, (c_date, data) in enumerate(evol_data.items()):
        f_rt  = data["f_rt"]; y_obs = data["y_obs"]; t_cut = len(y_obs)
        label = (f"Updated {pd.to_datetime(c_date).strftime('%b %Y')} "
                 f"(k={BEST_K_RT})")
        ax.plot(
            np.arange(t_cut - 1, t_cut - 1 + len(f_rt[t_cut-1:])),
            f_rt[t_cut-1:],
            color=colors_e[i], linestyle="--",
            linewidth=2.0, label=label, zorder=12+i)
        ax.scatter(t_cut - 1, y_obs[-1],
                   color=colors_e[i], s=55, zorder=20,
                   edgecolor="white", linewidth=1.0)
    ax.set_title("Retrospective Operational Evolution of the SC25 Forecast")
    ax.set_xlabel("Months since SC25 start")
    ax.set_ylabel("Smoothed SSN")
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", fontsize=9, frameon=False)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(make_fig11(), "fig11_sc25_forecast_evolution")

Manuscript Table 3:
    Cutoff  Months              Mode    Analogs  Peak SSN Peak Date
  Pre-SC25       0  Ref. shape (k=2)     19, 21     188.7   2024-02
2020-12-01      12 Obs. prefix (k=3) 17, 15, 19     208.8   2024-01
2021-12-01      24 Obs. prefix (k=3) 17, 15, 21     184.6   2024-01
2022-12-01      36 Obs. prefix (k=3) 17, 15, 21     177.5   2024-01
2023-12-01      48 Obs. prefix (k=3) 17, 15, 21     131.2   2024-01
2024-12-01      60 Obs. prefix (k=3) 17, 15, 21     173.4   2024-01


  saved: fig11_sc25_forecast_evolution.pdf / .eps


In [ ]:
# ============================================================
# Baseline SC26 forecast with 80% and 98% PI
# ============================================================
make_dual_pi_fig(
    f26_bl, lo80_bl26, hi80_bl26, lo98_bl26, hi98_bl26,
    title=f"Baseline SC26 forecast (k={BEST_K_BL}, "
          f"analogs SC{sel26_bl[0]} & SC{sel26_bl[1]})",
    stem="fig12_sc26_baseline_forecast_pi")

  saved: fig12_sc26_baseline_forecast_pi.pdf / .eps


In [ ]:
# ============================================================
# Combined two-cycle projection (SC25 + SC26)
# ============================================================
def make_fig13():
    fig, ax = plt.subplots(figsize=(8.0, 3.8))
    ax.plot(np.arange(len(f25_26_stitched)), f25_26_stitched,
            linewidth=1.8, color="#01579B")
    ax.axvline(len(f25_rt) - 1, linestyle="--",
               linewidth=1.0, color="#78909C")
    ax.set_title(
        "Combined Forecast for Solar Cycles 25 and 26 "
        "(Smooth Transition)")
    ax.set_xlabel("Months since SC25 start")
    ax.set_ylabel("Smoothed SSN")
    ax.set_ylim(bottom=0)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(make_fig13(), "fig13_sc25_sc26_stitched")

  saved: fig13_sc25_sc26_stitched.pdf / .eps


In [ ]:
# ============================================================
#  Leave-one-out sensitivity test
# ============================================================
loo_rows  = []
full_peak = float(np.max(f25_rt))
print(f"SC25 full-pool peak: {full_peak:.1f} SSN")
for excluded in odd_pool:
    reduced = [c for c in odd_pool if c != excluded]
    f_loo, sel_loo, _, _, _ = forecast_realtime(
        y_obs_25, reduced, BEST_K_RT)
    peak  = float(np.max(f_loo))
    delta = peak - full_peak
    loo_rows.append({
        "Excluded SC":excluded,
        "Analogs selected":str(sel_loo),
        "Peak SSN":round(peak, 1),
        "Δ vs full pool":round(delta, 1),
    })
    print(f"  Excl. SC{excluded}: peak={peak:.1f}  Δ={delta:+.1f}  "
          f"analogs={sel_loo}")

loo_df = pd.DataFrame(loo_rows)
loo_df.to_csv(os.path.join(OUT_TAB, "table_loo_sensitivity.csv"),
              index=False)
max_delta = loo_df["Δ vs full pool"].abs().max()
print(f"  Max |Δ| = {max_delta:.1f} SSN  (manuscript: 7.4)")

def make_fig14():
    fig, ax = plt.subplots(figsize=(8.0, 3.8))
    x = np.arange(len(loo_rows))
    ax.bar(x, [r["Peak SSN"] for r in loo_rows],
           color="steelblue", alpha=0.85)
    ax.axhline(full_peak, color="black",
               linewidth=1.5, linestyle="--",
               label=f"Full pool: {full_peak:.1f} SSN")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"Excl.\nSC{r['Excluded SC']}" for r in loo_rows],
        fontsize=9)
    ax.set_ylabel("Forecast peak SSN")
    ax.set_title(
        "SC25 Forecast Stability: Leave-One-Out Analog Pool Test\n"
        "(each bar = full odd-cycle pool minus one cycle)")
    ax.legend(frameon=False)
    ax.set_ylim(0)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(make_fig14(), "fig14_loo_sensitivity")

SC25 full-pool peak: 173.4 SSN
  Excl. SC13: peak=173.4  Δ=+0.0  analogs=[17, 15, 21]
  Excl. SC15: peak=167.3  Δ=-6.0  analogs=[17, 21, 13]
  Excl. SC17: peak=172.9  Δ=-0.4  analogs=[15, 21, 13]
  Excl. SC19: peak=173.4  Δ=+0.0  analogs=[17, 15, 21]
  Excl. SC21: peak=165.9  Δ=-7.4  analogs=[17, 15, 13]
  Excl. SC23: peak=173.4  Δ=+0.0  analogs=[17, 15, 21]
  Max |Δ| = 7.4 SSN  (manuscript: 7.4)


  saved: fig14_loo_sensitivity.pdf / .eps


In [ ]:
# ============================================================
# TABLES 2 and 4
# ============================================================

# Table 2 — SC25 forecast summary
mae25_rt_full = float(mean_absolute_error(y_obs_25, f25_rt[:t_len]))
rmse25_rt_full = true_rmse(y_obs_25, f25_rt[:t_len])
table2 = pd.DataFrame([
    {
        "Mode":"Baseline",
        "k":BEST_K_BL,
        "Analogs":str(sel25_bl),
        "Peak SSN":round(float(np.max(f25_bl)), 1),
        "Peak Month":int(np.argmax(f25_bl)),
        "MAE":"—","RMSE":"—","R2":"—",
        "C80":"—","C98":"—",
    },
    {
        "Mode":"Real-time",
        "k":BEST_K_RT,
        "Analogs":str(sel25_rt),
        "Peak SSN":round(float(np.max(f25_rt)), 1),
        "Peak Month":int(np.argmax(f25_rt)),
        "MAE":round(mae25_rt_full, 2),
        "RMSE":round(rmse25_rt_full, 2),
        "R2":round(r2_25_rt, 3),
        "C80":f"{100*cov80_rt:.1f}%",
        "C98":f"{100*cov98_rt:.1f}%",
    },
])
table2.to_csv(os.path.join(OUT_TAB, "table2_sc25_forecasts.csv"),
              index=False)
print("Manuscript Table 2:")
print(table2.to_string(index=False))

# Table 4 — SC26 baseline forecast
table4 = pd.DataFrame([{
    "k":BEST_K_BL,
    "Analogs":str(sel26_bl),
    "Peak SSN":round(float(np.max(f26_bl)), 1),
    "Peak Month":pk26,
    "80% PI lo":round(lo80_bl26[pk26], 1),
    "80% PI hi":round(hi80_bl26[pk26], 1),
    "98% PI lo":round(lo98_bl26[pk26], 1),
    "98% PI hi":round(hi98_bl26[pk26], 1),
}])
table4.to_csv(os.path.join(OUT_TAB, "table4_sc26_baseline.csv"),
              index=False)
print("\nManuscript Table 4:")
print(table4.to_string(index=False))

Manuscript Table 2:
     Mode  k      Analogs  Peak SSN  Peak Month    MAE   RMSE     R2   C80    C98
 Baseline  2     [19, 21]     188.7          49      —      —      —     —      —
Real-time  3 [17, 15, 21]     173.4          48  12.28  16.37  0.908 88.3% 100.0%

Manuscript Table 4:
 k  Analogs  Peak SSN  Peak Month  80% PI lo  80% PI hi  98% PI lo  98% PI hi
 2 [18, 22]     143.3          41      119.2      173.2       83.5      210.5


In [ ]:
# ============================================================
# §11.1  DTW BAND CONSTRAINT SENSITIVITY
#
# Tests whether Sakoe-Chiba band constraints or exact unconstrained
# DTW change the validation MAE or SC25 analog selection.
#
# ALL experiments use the REAL SILSO v2.0 cycle library,
#
# Implements unconstrained exact DTW and constrained variants
# (10%, 20%, 30% of input length) alongside the published fastdtw.
# ============================================================

def dtw_exact(x, y):
    """Exact unconstrained DTW — O(nm) dynamic programming."""
    n, m = len(x), len(y)
    D    = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = abs(x[i-1] - y[j-1]) + min(
                D[i-1, j], D[i, j-1], D[i-1, j-1])
    return float(D[n, m])

def dtw_sakoe_chiba(x, y, window):
    """Constrained DTW with Sakoe-Chiba band (Sakoe & Chiba 1978)."""
    n, m = len(x), len(y)
    w    = max(window, abs(n - m))   # guarantee feasibility
    D    = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(max(1, i - w), min(m, i + w) + 1):
            D[i, j] = abs(x[i-1] - y[j-1]) + min(
                D[i-1, j], D[i, j-1], D[i-1, j-1])
    return float(D[n, m])

def dtw_select_custom(target_curve, pool_cycles, k, dtw_fn):
    """dtw_select with a pluggable distance function."""
    t = _norm01(target_curve)
    dist_list = []
    for c in pool_cycles:
        yc     = _norm01(cycle_fits[c][1])
        yc_adj = np.interp(np.linspace(0,1,len(t)),
                            np.linspace(0,1,len(yc)), yc)
        dist_list.append((c, dtw_fn(t, yc_adj)))
    dist_list.sort(key=lambda z: z[1])
    chosen = dist_list[:min(k, len(dist_list))]
    sel    = [c for c, _ in chosen]
    eps    = 1e-9
    w      = np.array([1.0 / (d + eps) for _, d in chosen], float)
    w     /= w.sum()
    return sel, w, dist_list

def eval_rt_custom_dtw(sc, k, dtw_fn):
    """Walk-forward RT error using a custom DTW function."""
    _, y = cycle_fits[sc]
    pool = [c for c in range(13, sc) if c % 2 == sc % 2]
    m    = min(PREFIX_LEN, len(y))
    sel, w, _ = dtw_select_custom(y[:m], pool, k, dtw_fn)
    target_len = max(m, int(round(
        np.mean([len(cycle_fits[c][1]) for c in sel]))))
    sh   = blend_shape_norm(sel, w, target_len)
    f    = apply_idw_scaling(sh, sel, w)
    f   += (y[m-1] - f[m-1])
    f    = np.clip(f, 0.0, None)
    n_m  = min(len(y), len(f))
    return float(mean_absolute_error(y[:n_m], f[:n_m]))

Q = PREFIX_LEN   # 60 months — query length used in real-time mode

dtw_variants = {
    "fastdtw (current, r=1)":
        (lambda x, y: float(fastdtw(x, y)[0])),
    "Unconstrained exact DTW":
        dtw_exact,
    f"SC band 10% (w={int(0.10*Q)})":
        (lambda x, y: dtw_sakoe_chiba(x, y, int(0.10*Q))),
    f"SC band 20% (w={int(0.20*Q)})":
        (lambda x, y: dtw_sakoe_chiba(x, y, int(0.20*Q))),
    f"SC band 30% (w={int(0.30*Q)})":
        (lambda x, y: dtw_sakoe_chiba(x, y, int(0.30*Q))),
}

b1_rows = []
print("Band constraint experiment (real SILSO data):")
for label, fn in dtw_variants.items():
    maes = [eval_rt_custom_dtw(sc, BEST_K_RT, fn)
            for sc in VAL_CYCLES]
    mean_mae = float(np.mean(maes))
    # SC25 selection
    sel_sc25, _, _ = dtw_select_custom(
        y_obs_25, odd_pool, BEST_K_RT, fn)
    b1_rows.append({
        "DTW variant":label,
        "Avg MAE":round(mean_mae, 2),
        "SC25 analogs":str(sel_sc25),
    })
    print(f"  {label:<38}: MAE={mean_mae:.2f}  analogs={sel_sc25}")

b1_df = pd.DataFrame(b1_rows)
b1_df.to_csv(os.path.join(OUT_TAB,
    "table_supp_dtw_band_sensitivity.csv"), index=False)

ref_mae = b1_df.loc[0, "Avg MAE"]
max_diff = b1_df["Avg MAE"].sub(ref_mae).abs().max()
all_same = all(b1_df["SC25 analogs"] == b1_df.loc[0, "SC25 analogs"])
print()
print("  INTERPRETATION (Section 3.3):")
print(f"  SC25 analog selection identical across all DTW variants: {all_same}")
print(f"  Validation MAE varies across band constraints "
      f"(max diff vs fastdtw: {max_diff:.2f} sunspots).")

Band constraint experiment (real SILSO data):
  fastdtw (current, r=1)                : MAE=20.43  analogs=[17, 15, 21]
  Unconstrained exact DTW               : MAE=17.99  analogs=[17, 15, 21]
  SC band 10% (w=6)                     : MAE=24.12  analogs=[17, 15, 21]
  SC band 20% (w=12)                    : MAE=26.89  analogs=[17, 15, 21]
  SC band 30% (w=18)                    : MAE=21.78  analogs=[17, 15, 21]

  INTERPRETATION (Section 3.3):
  SC25 analog selection identical across all DTW variants: True
  Validation MAE varies across band constraints (max diff vs fastdtw: 6.46 sunspots).


In [ ]:
# ============================================================
# §11.2  VARIANCE FLOOR (ν) SENSITIVITY
#
# Examines how the choice of ν in Eq. (10) affects PI width
# at solar maximum and minimum, and whether it changes
# empirical coverage on the held-out SC25 observed window.
#
# Uses REAL f25_rt computed from real SILSO data.
# ============================================================

f_pos_rt = np.clip(f25_rt, 0.0, None)
f_max_rt = float(np.max(f_pos_rt))
ramp_raw = np.sqrt(f_pos_rt / f_max_rt)
pk_rt_idx = int(np.argmax(f25_rt))
t_min_idx = int(np.argmin(f25_rt[:t_len]))  # within observed window

nu_grid   = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]
b2_rows   = []

print("Variance floor sensitivity (real SC25 forecast):")
print(f"  {'ν':>6}  {'floor active (months)':>22}  "
      f"{'PI_peak':>10}  {'PI_min':>8}  {'coverage80':>12}")

for nu in nu_grid:
    n_active = int(np.sum(ramp_raw < nu))
    lo80_nu, hi80_nu = bootstrap_pi(
        f25_rt, odd_bins_80, q=(0.10, 0.90),
        seed=RNG_SEED + 80, nu=nu)
    w_peak = float(hi80_nu[pk_rt_idx] - lo80_nu[pk_rt_idx])
    w_min  = float(hi80_nu[t_min_idx] - lo80_nu[t_min_idx])
    cov80_nu = float(np.mean(
        (y_obs_25 >= lo80_nu[:t_len]) &
        (y_obs_25 <= hi80_nu[:t_len])))
    note = " ← published" if abs(nu - NU_PUB) < 0.001 else ""
    b2_rows.append({
        "ν":nu,
        "floor_active_months":n_active,
        "PI_width_at_peak":round(w_peak, 1),
        "PI_width_at_min":round(w_min, 1),
        "SC25_coverage_80pct":round(cov80_nu, 4),
    })
    print(f"  {nu:>6.2f}  {n_active:>22d}  "
          f"{w_peak:>10.1f}  {w_min:>8.1f}  "
          f"{100*cov80_nu:>11.1f}%{note}")

b2_df = pd.DataFrame(b2_rows)
b2_df.to_csv(os.path.join(OUT_TAB,
    "table_supp_nu_sensitivity.csv"), index=False)

peak_range = b2_df["PI_width_at_peak"].max() - b2_df["PI_width_at_peak"].min()
cov_range  = (b2_df["SC25_coverage_80pct"].max()
              - b2_df["SC25_coverage_80pct"].min())

print()


Variance floor sensitivity (real SC25 forecast):
       ν   floor active (months)     PI_peak    PI_min    coverage80
    0.15                      28        65.5       9.5         76.7%
    0.20                      31        65.5      12.7         86.7%
    0.25                      34        65.5      15.9         88.3%
    0.30                      37        65.5      19.0         88.3%
    0.35                      40        65.5      22.2         88.3% ← published
    0.40                      43        65.5      25.4         88.3%
    0.45                      45        65.5      28.5         88.3%
    0.50                      52        65.5      31.7         88.3%
    0.55                      58        65.5      34.9         88.3%



In [ ]:
# ============================================================
# §11.3  Prefix-length instability table
# ============================================================
prefix_rows = []
print("SC25 peak estimate by observation prefix length (Section 4.5):")
for plen in [12, 24, 36, 48, 60]:
    obs = y_obs_25[:plen]
    f, sel, _, _, _ = forecast_realtime(obs, odd_pool, BEST_K_RT)
    f    = np.clip(f, 0.0, None)
    peak = float(np.max(f))
    prefix_rows.append({
        "Prefix months":plen,
        "Peak SSN":round(peak, 1),
        "Peak month":int(np.argmax(f)),
        "Analogs":str(sel),
    })
    print(f"  {plen:2d}m: peak={peak:.1f}  analogs={sel}")

prefix_df = pd.DataFrame(prefix_rows)
prefix_df.to_csv(
    os.path.join(OUT_TAB, "table_supp_prefix_instability.csv"),
    index=False)

SC25 peak estimate by observation prefix length (Section 4.5):
  12m: peak=208.8  analogs=[17, 15, 19]
  24m: peak=184.6  analogs=[17, 15, 21]
  36m: peak=177.5  analogs=[17, 15, 21]
  48m: peak=131.2  analogs=[17, 15, 21]
  60m: peak=173.4  analogs=[17, 15, 21]
